In [1]:
import numpy as np
import scipy
from scipy.optimize import curve_fit
import math
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import healpy as hp

import pickle
plt = sns.mpl.pyplot
from scipy.optimize import fmin_l_bfgs_b

import h5py as h5
import h5py

import json

# plotting the spec signal data at different temperatures:
import numpy as np
import scipy
from scipy import special

import sys

from datetime import datetime
import pytz

import time


In [2]:
'''
num = str(11)

hem = '/home/ecp/disk1/pocam_data/batch1/prod_characterization_'+num+'/cali_flange_'+num

h = h5py.File(hem, 'r')

print(h.keys(),'\n')
print(h['meta'].attrs.keys(), '\n')



for key in list(h['meta'].attrs.keys()):
    print(key, ' :   ',  h['meta'].attrs[key])

'''

"\nnum = str(11)\n\nhem = '/home/ecp/disk1/pocam_data/batch1/prod_characterization_'+num+'/cali_flange_'+num\n\nh = h5py.File(hem, 'r')\n\nprint(h.keys(),'\n')\nprint(h['meta'].attrs.keys(), '\n')\n\n\n\nfor key in list(h['meta'].attrs.keys()):\n    print(key, ' :   ',  h['meta'].attrs[key])\n\n"

In [3]:
date = h['meta'].attrs['date']

new_date = ''
for ymd in date.split('-'):
    new_date += ymd
print(new_date)
# Step 1: Parse the naive datetime (no timezone yet)
dt_naive = datetime.strptime(date, '%Y-%m-%d_%H_%M_%S')
# Step 2: Localize to Europe/Berlin timezone (handles DST)
munich_tz = pytz.timezone('Europe/Berlin')
dt_local = munich_tz.localize(dt_naive)
# Step 3: Convert to UTC
dt_utc = dt_local.astimezone(pytz.utc)
# Step 4: Convert to Unix timestamp
unix_time = dt_utc.timestamp()
new_meas_time = unix_time
print("Unix time:", unix_time)

NameError: name 'h' is not defined

## Classes and Functions to Load and Process The Raw Data for Isotropy Measurements

In [101]:
class isotropy_value_batch1_old:
    
    def __init__(self,
                 hemispheres = ['03', '04'],
                 diode = 'LMG405'):
        
        self.hemispheres = hemispheres
        self.diode = diode
        
        """ This function is to determine how isotropic a single hemisphere or also one complete POCAM can emit light.
            The exact value depends on the chosen convention. So far we use normalize the angular emission to the mean
            and take the average (which is effectively dividing the sum of absolute deviations from the mean by 2)
            of the maximum and minimum deviation. It is used to determine how isotropic one POCAM device or
            how isotropic one single hemisphere can flash. """

        # led = ['LMG405', 'LMG520', 'KAPU465', 'LMG450', 'KAPU405', 'LMG365']

        hem1 = f'/home/ecp/disk1/pocam_data/batch1/prod_characterization_{self.hemispheres[0]}/cali_flange_{self.hemispheres[0]}'
        hem2 = f'/home/ecp/disk1/pocam_data/batch1/prod_characterization_{self.hemispheres[1]}/cali_flange_{self.hemispheres[1]}'
        
        
        #flange1 = single(hem1)
        #flange2 = single(hem2)
        #flangea = single(hema)
        #flange3 = single(hem3)

        ha = h5py.File(hem1, 'r')
        hb = h5py.File(hem2, 'r')
        
        self.date1 = ha['meta'].attrs['date']
        self.date2 = hb['meta'].attrs['date']
        

        new_date = ''
        for ymd in self.date1.split('-'):
            new_date += ymd
        #print(new_date)
        # Step 1: Parse the naive datetime (no timezone yet)
        dt_naive = datetime.strptime(self.date1, '%Y-%m-%d_%H_%M_%S')
        # Step 2: Localize to Europe/Berlin timezone (handles DST)
        munich_tz = pytz.timezone('Europe/Berlin')
        dt_local = munich_tz.localize(dt_naive)
        # Step 3: Convert to UTC
        dt_utc = dt_local.astimezone(pytz.utc)
        # Step 4: Convert to Unix timestamp
        unix_time = dt_utc.timestamp()
        self.meas_time = unix_time
        
        
        fl1 = h5py.File(hem1,'r')
        fl2 = h5py.File(hem2,'r')
        #fla = h5py.File(hema,'r')
        #fl3 = h5py.File(hem3,'r')

        h1=[]
        h2=[]

        data1 = np.array(fl1.get(diode))
        data2 = np.array(fl2.get(diode))
        zen=fl1['meta'].attrs['zenith']
        azi=fl1['meta'].attrs['azimuth']
        nzen=fl1['meta'].attrs['n_zenith']
        nazi=fl1['meta'].attrs['n_azimuth']
        szen=fl1['meta'].attrs['s_zenith']
        sazi=fl1['meta'].attrs['s_azimuth']

        azi = azi.reshape(16,6)
        zen = zen.reshape(16,6)

        # adjust zenith angles
        a = [170.0,180.0,190.0]
        addzen=np.reshape(np.repeat(160.0,zen.shape[1]+1),(1,-1))
        zen=np.append(zen,zen[:,0][...,None],axis=1)
        zen=np.append(zen,addzen,axis=0)
        for k in a:
            addzen=np.reshape(np.repeat(k,zen.shape[1]),(1,-1))
            zen=np.append(zen,addzen,axis=0)
        # print(zen.shape)

        # adjust azimuth angles
        app = azi[-1]
        for k in range(len(a)+1):
            azi = np.append(azi,app)
        azi = azi.reshape(20,6)

        addazi=np.reshape(np.repeat(nazi*sazi,azi.shape[0]+1),(-1,1))
        azi=np.append(azi,azi[0,:][None,...],axis=0)
        azi=np.append(azi,addazi,axis=1)
        azi = azi[:-1]
        # print(azi.shape)

        zen=90-zen
        azi=azi-180

        zen=np.radians(zen)
        azi=np.radians(azi)


        matching_factor = np.mean(data1[0,0,:]) / np.mean(data2[0,0,:])

        data1=data1[0,:,:]#/data[0,0,0]


        data2=data2[0,:,:] * float(matching_factor)



        weight_air = np.array([1,1,1,1,1,1,0.97,0.95,0.79,0.55,0.3,0.09,1,1,1,1])
        weight_ice = np.array([1,1,1,1,1,1,0.97,0.95,0.8,0.5,0.18,0.1,1,1,1,1])

        weight = weight_ice/weight_air


        for k in range(6):
            data1[:,k] = data1[:,k]*weight
            data2[:,k] = data2[:,k]*weight# *diff

        # print(diff)


        data_add = np.zeros((len(a),6))
        data1 = np.concatenate((data1,data_add),axis=0)
        data2 = np.concatenate((data2,data_add),axis=0)

        # print(data.shape)

        data2 = np.flip(data2)


        diff_data = np.add(data1,data2)


        mean = np.mean(np.mean(diff_data, axis=0))
        # print(mean)

        #maximum = np.max(diff_data/mean)
        ##print('max: ', np.max(diff_data/mean))

        #minimum = np.min(diff_data/mean)
        ##print('min: ', np.min(diff_data/mean))

        single_values = diff_data/mean
        self.y_values = single_values.tolist()
        self.isotropy_value = ( np.max(single_values) - np.min(single_values) ) / 2
        self.isotropy_error = None
            
            

In [3]:
def sigmoid(x, k, w, x0, y0):
    return k / (1. + np.exp(-w * (x - x0))) + y0 

In [11]:
class isotropy_value_batch1:
    
    def __init__(self,
                 hemispheres = ['03', '04'],
                 diode = 'LMG405'):
        
        self.hemispheres = hemispheres
        self.diode = diode
        
        """ This function is to determine how isotropic a single hemisphere or also one complete POCAM can emit light.
            The exact value depends on the chosen convention. So far we use normalize the angular emission to the mean
            and take the average (which is effectively dividing the sum of absolute deviations from the mean by 2)
            of the maximum and minimum deviation. It is used to determine how isotropic one POCAM device or
            how isotropic one single hemisphere can flash. """

        # led = ['LMG405', 'LMG520', 'KAPU465', 'LMG450', 'KAPU405', 'LMG365']

        hem1 = f'/home/ecp/disk1/pocam_data/batch1/prod_characterization_{self.hemispheres[0]}/cali_flange_{self.hemispheres[0]}'
        hem2 = f'/home/ecp/disk1/pocam_data/batch1/prod_characterization_{self.hemispheres[1]}/cali_flange_{self.hemispheres[1]}'
        
        
        #flange1 = single(hem1)
        #flange2 = single(hem2)
        #flangea = single(hema)
        #flange3 = single(hem3)

        ha = h5py.File(hem1, 'r')
        hb = h5py.File(hem2, 'r')
        
        self.date1 = ha['meta'].attrs['date']
        self.date2 = hb['meta'].attrs['date']
        

        new_date = ''
        for ymd in self.date1.split('-'):
            new_date += ymd
        #print(new_date)
        # Step 1: Parse the naive datetime (no timezone yet)
        dt_naive = datetime.strptime(self.date1, '%Y-%m-%d_%H_%M_%S')
        # Step 2: Localize to Europe/Berlin timezone (handles DST)
        munich_tz = pytz.timezone('Europe/Berlin')
        dt_local = munich_tz.localize(dt_naive)
        # Step 3: Convert to UTC
        dt_utc = dt_local.astimezone(pytz.utc)
        # Step 4: Convert to Unix timestamp
        unix_time = dt_utc.timestamp()
        self.meas_time = unix_time
        
        
        fl1 = h5py.File(hem1,'r')
        fl2 = h5py.File(hem2,'r')
        #fla = h5py.File(hema,'r')
        #fl3 = h5py.File(hem3,'r')

        h1=[]
        h2=[]

        data_1 = np.array(fl1.get(diode))
        data_2 = np.array(fl2.get(diode))
        zen=fl1['meta'].attrs['zenith']

        zen = zen.reshape(16,6)
        
        zen_orig = zen
    
    
        data1 = data_1[0,:,:]
        data2 = data_2[0,:,:]
        
        err1 = data_1[1,:,:]
        err2 = data_2[1,:,:]

        
        #print(data1)
        
        mini1=np.min(data1)
        mini2=np.min(data2)
        
        
        y1=data1/mini1
        y2=data2/mini2
        
        err1=err1/mini1
        err2=err2/mini2
        
        
        #matching_factor = np.mean(data1[0,0,:]) / np.mean(data2[0,0,:])
        #data1=data1[0,:,:]#/data[0,0,0]
        #data2=data2[0,:,:] * float(matching_factor)


        
        x_pre = np.array([62.70,60.16,57.69,55.16,52.62,50.22,47.54,45.14,42.67,40.27,37.66,34.99,32.59,30.05,27.58,25.18,22.64,20.24,17.63,15.09,12.69,10.08,7.54,5.07,2.67,0.34])
        y_pre = np.array([0.960,0.966,0.968,0.969,0.971,0.975,0.980,0.981,0.990,0.990,0.991,0.991,0.994,0.997,1.000,1.000,1.000,1.003,1.001,1.004,1.004,1.006,1.006,1.006,1.006,1.007])
        x_values = np.array([65.17,67.64,70.18,70.11,72.64,72.58,75.18,75.25,77.64,77.65,80.18,80.18,82.64,82.64,85.10,85.10,87.64,87.63,90.10,90.16,90.26,92.62,95.09,95.08,97.56,97.55,100.16,100.01,102.55,102.54,105.01,105.00,107.55,107.54,110.01,110.07,112.61,112.54,115.08,115.15,117.62,117.62,119.95,119.95,125.,125.,130.,130.,135.,135.,140.,140.,150.,150.,160.,160.])
        y_values = np.array([0.958,0.953,0.939,0.949,0.909,0.934,0.874,0.903,0.832,0.860,0.783,0.798,0.725,0.726,0.668,0.656,0.606,0.577,0.542,0.497,0.480,0.416,0.416,0.337,0.353,0.262,0.293,0.186,0.233,0.122,0.176,0.069,0.125,0.025,0.075,0.005,0.039,-0.001,0.014,-0.001,0.006,-0.002,0.005,-0.002,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.])
        x_air = np.concatenate([np.flip(x_pre), x_values[::2]])[10:]
        x_ice = np.concatenate([np.flip(x_pre), x_values[1::2]])
        y_air = np.concatenate([np.flip(y_pre), y_values[::2]])[10:]
        y_ice = np.concatenate([np.flip(y_pre), y_values[1::2]])

        popt_air, _ = curve_fit(sigmoid, x_air, y_air, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)
        popt_ice, _ = curve_fit(sigmoid, x_ice, y_ice, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)

        
        #print(np.degrees(zen))
        #print(y1)

        
        y1 = y1*sigmoid(zen_orig,*popt_ice)/sigmoid(zen_orig,*popt_air)
        y2 = y2*sigmoid(zen_orig,*popt_ice)/sigmoid(zen_orig,*popt_air)
        
        err1 = err1*sigmoid(zen_orig,*popt_ice)/sigmoid(zen_orig,*popt_air)
        err2 = err2*sigmoid(zen_orig,*popt_ice)/sigmoid(zen_orig,*popt_air)

        
        data_add = np.zeros((3,6))
        y1 = np.concatenate((y1,data_add),axis=0)
        y2 = np.concatenate((y2,data_add),axis=0)
        
        err1 = np.concatenate((err1,data_add),axis=0)
        err2 = np.concatenate((err2,data_add),axis=0)
        

        # print(data.shape)

        y_mirror = np.flip(y2)
        y_mirror_err = np.flip(err2)

        
        y_total = (y1 + y_mirror)
        y_total_err = (err1 + y_mirror_err)
        
        
        
        y_values = y_total/np.mean(y_total)
        
        #print(y_values)
        
       
        N = y_total.shape[0]
        #print(N)
        
        # sum of all variances (needed for cross terms)
        total_var = np.sum(y_total_err**2)
        # propagated errors
        y_err = np.sqrt(
            ((np.mean(y_total) - y_total / N) / np.mean(y_total)**2)**2 * y_total_err**2
            + ((y_total / (N * np.mean(y_total)**2))**2) * (total_var - y_total_err**2) )
        
        
        self.y_values = list(y_values)
        
        i_max = np.unravel_index(np.argmax(y_values, axis=None), y_values.shape)
        i_min = np.unravel_index(np.argmin(y_values, axis=None), y_values.shape)

        self.isotropy_value = (y_values[i_max] - y_values[i_min]) / 2
        self.isotropy_error = np.sqrt(y_err[i_max]**2 + y_err[i_min]**2)/2
    


        
        #self.isotropy_value = ( np.max(y_values) - np.min(y_values) ) / 2
        #self.isotropy_error = None
            

In [12]:
# Isotropy of both hemispheres of one POCAM device:


class isotropy_value_batch2:
    
    def __init__(self,
                 hemispheres = ['03', '04'],
                 diode = 'LMG405'):
        
        self.hemispheres = hemispheres
        self.diode = diode
    
        """ This function is to determine how isotropic a single hemisphere or also one complete POCAM can emit light.
            The exact value depends on the chosen convention. So far we use normalize the angular emission to the mean
            and take the average (which is effectively dividing the sum of absolute deviations from the mean by 2)
            of the maximum and minimum deviation. It is used to determine how isotropic one POCAM device or
            how isotropic one single hemisphere can flash. """

        # led = ['LMG405', 'LMG520', 'KAPU465', 'LMG450', 'KAPU405', 'LMG365']

        hem1 = f'/home/ecp/disk1/pocam_data/batch2/prod_characterization_{self.hemispheres[0]}/cali_flange_{self.hemispheres[0]}'
        hem2 = f'/home/ecp/disk1/pocam_data/batch2/prod_characterization_{self.hemispheres[1]}/cali_flange_{self.hemispheres[1]}'

        NSIDE = 2**2
        NPIX = hp.nside2npix(NSIDE)

        vec = hp.ang2vec(np.pi / 2, np.pi * 3 / 4)
        ipix_disc = hp.query_strip(NSIDE, np.radians(0), np.radians(150))
        deg = np.degrees(hp.pix2ang(nside=NSIDE, ipix=ipix_disc))


        x_pre = np.array([62.70,60.16,57.69,55.16,52.62,50.22,47.54,45.14,42.67,40.27,37.66,34.99,32.59,30.05,27.58,25.18,22.64,20.24,17.63,15.09,12.69,10.08,7.54,5.07,2.67,0.34])
        y_pre = np.array([0.960,0.966,0.968,0.969,0.971,0.975,0.980,0.981,0.990,0.990,0.991,0.991,0.994,0.997,1.000,1.000,1.000,1.003,1.001,1.004,1.004,1.006,1.006,1.006,1.006,1.007])
        x_values = np.array([65.17,67.64,70.18,70.11,72.64,72.58,75.18,75.25,77.64,77.65,80.18,80.18,82.64,82.64,85.10,85.10,87.64,87.63,90.10,90.16,90.26,92.62,95.09,95.08,97.56,97.55,100.16,100.01,102.55,102.54,105.01,105.00,107.55,107.54,110.01,110.07,112.61,112.54,115.08,115.15,117.62,117.62,119.95,119.95,125.,125.,130.,130.,135.,135.,140.,140.,150.,150.,160.,160.])
        y_values = np.array([0.958,0.953,0.939,0.949,0.909,0.934,0.874,0.903,0.832,0.860,0.783,0.798,0.725,0.726,0.668,0.656,0.606,0.577,0.542,0.497,0.480,0.416,0.416,0.337,0.353,0.262,0.293,0.186,0.233,0.122,0.176,0.069,0.125,0.025,0.075,0.005,0.039,-0.001,0.014,-0.001,0.006,-0.002,0.005,-0.002,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.])
        x_air = np.concatenate([np.flip(x_pre), x_values[::2]])[10:]
        x_ice = np.concatenate([np.flip(x_pre), x_values[1::2]])
        y_air = np.concatenate([np.flip(y_pre), y_values[::2]])[10:]
        y_ice = np.concatenate([np.flip(y_pre), y_values[1::2]])

        popt_air, _ = curve_fit(sigmoid, x_air, y_air, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)
        popt_ice, _ = curve_fit(sigmoid, x_ice, y_ice, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)

        h1 = h5py.File(hem1, 'r')
        h2 = h5py.File(hem2, 'r')
        
        self.date1 = h1['meta'].attrs['date']
        self.date2 = h2['meta'].attrs['date']
        

        new_date = ''
        for ymd in self.date1.split('-'):
            new_date += ymd
        #print(new_date)
        # Step 1: Parse the naive datetime (no timezone yet)
        dt_naive = datetime.strptime(self.date1, '%Y-%m-%d_%H_%M_%S')
        # Step 2: Localize to Europe/Berlin timezone (handles DST)
        munich_tz = pytz.timezone('Europe/Berlin')
        dt_local = munich_tz.localize(dt_naive)
        # Step 3: Convert to UTC
        dt_utc = dt_local.astimezone(pytz.utc)
        # Step 4: Convert to Unix timestamp
        unix_time = dt_utc.timestamp()
        self.meas_time = unix_time
        
        
        data1 = np.array(h1.get(self.diode)[2])
        data2 = np.array(h2.get(self.diode)[2])
        
        err1 = np.array(h1.get(self.diode)[3])
        err2 = np.array(h2.get(self.diode)[3])
        


        # adjust azimuth angles
        mini1 = min(data1)
        mini2 = min(data2)
            

        y1 = np.array(data1)/mini1
        y2 = np.array(data2)/mini2
        
        y_err1 = np.array(err1)/mini1
        y_err2 = np.array(err2)/mini2

        
        
        y1 = y1*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)
        y2 = y2*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)
        
        y_err1 = y_err1*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)
        y_err2 = y_err2*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)
        

        
        
        y1 = np.concatenate((y1,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        y2 = np.concatenate((y2,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        
        y_err1 = np.concatenate((y_err1,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        y_err2 = np.concatenate((y_err2,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        
        

        y_mirror = np.flip(y2)
        y_mirror_err = np.flip(y_err2)
        
        
        y_total = (y1 + y_mirror)
        y_total_err = (y_err1 + y_mirror_err)
        
        
        y_values = y_total/np.mean(y_total)
        
        
        N = y_total.shape[0]
        # sum of all variances (needed for cross terms)
        total_var = np.sum(y_total_err**2)
        # propagated errors
        y_err = np.sqrt(
            ((np.mean(y_total) - y_total / N) / np.mean(y_total)**2)**2 * y_total_err**2
            + ((y_total / (N * np.mean(y_total)**2))**2) * (total_var - y_total_err**2) )
        
        
        self.y_values = list(y_values)
        
        i_max = np.argmax(y_values)
        i_min = np.argmin(y_values)

        self.isotropy_value = (y_values[i_max] - y_values[i_min]) / 2
        self.isotropy_error = np.sqrt(y_err[i_max]**2 + y_err[i_min]**2)/2
        
        
        #self.isotropy_value = (np.max(y_values) - np.min(y_values))/2
        #self.isotropy_error = None
        


In [20]:
devices = {'003': [['11','12'], 'batch1'],'005': [['18','20'], 'batch1'],'006': [['19','22'], 'batch1'],
           '008': [['23','24'], 'batch1'],'010': [['09','10'], 'batch1'],'011': [['13','16'], 'batch1'],
           '002': [['03','04'], 'batch2'],'004': [['05','06'], 'batch2'],'009': [['14','15'], 'batch2'],
           '012': [['17','21'], 'batch2'],'013': [['25','26'], 'batch2'],'014': [['27','28'], 'batch2'],
           '015': [['29','30'], 'batch2'],'016': [['31','32'], 'batch2'],'017': [['33','34'], 'batch2'],
           '018': [['35','36'], 'batch2'],'019': [['37','38'], 'batch2'],'020': [['39','40'], 'batch2'],
           '021': [['41','42'], 'batch2'],'022': [['43','44'], 'batch2'],'023': [['45','46'], 'batch2'],
           '024': [['47','48'], 'batch2'],'025': [['49','50'], 'batch2'],'026': [['51','52'], 'batch2'],
           '027': [['53','54'], 'batch2'],'028': [['55','56'], 'batch2'] }

In [18]:
dioden = ['LMG365', 'LMG405', 'LMG450', 'LMG520', 'KAPU405', 'KAPU465']   # our standard form of 'LMG405', 'KAPU465', etc.

#pocam_device_number = '18'    # needs to be of the form '001', '002', ... , '019', etc.

pocam_ids = list(devices.keys())
#hemispheres = devices[pocam_ids][0]
#batch = devices[pocam_ids][1]



#target = 'master'             # 'master' or 'slave'
target = 'None'



temp = 25    
coarse = 1
fine = 20
mode = 'default'
pwm = 54000

device = '018'
hemispheres = devices[device][0]
batch = devices[device][1]

#print(hemispheres)



    
    
    
def file_creation(device_id, emitter):
    
    if 'LMG' in emitter:
        driver = 'l'               # 'l' for lmg
    else:
        driver = 'k'               # 'k' for kapu

        
    hemispheres = devices[device_id][0]
    
    batch = devices[device_id][1]

    isotropy = {}     # this is the dict to be stored as the Database json file for the spectral data for one L(E)D

    # Setting up the dict to be stored as json file:


    isotropy["device_uid"] = ''        # "pocam-20240405_001"
    isotropy["subdevice_uid"] = ''     # "pocam-led-{target}_{driver}-{emitter[-3:]}_{pocam_device_number}"
    isotropy["meas_name"] = "led-isotropy-level"
    isotropy["meas_class"] = "display"
    isotropy["meas_stage"] = "calibration"
    isotropy["meas_group"] = "isotropy"
    isotropy["meas_site"] = "tum"
    isotropy["meas_data"] = []



    if batch=='batch2':

        # Running the actual data inference:
        iso = isotropy_value_batch2(hemispheres=hemispheres,
                             diode= emitter)  


        ### take date of assembly of the device, as there will probably only one angular_emission_profile file per device
        ### more than one would mean, copy the same file in all subfolders

        ###### TALK TO MATT KAUER ABOUT THAT, also ask about subdevice_uid as there will be no master/slave


        # specifying some more key-value pairs for identification etc.:
        isotropy["device_uid"] = f"pocam-{iso.date1}_{device}"
        #if iso.target == '1':
        #    target = 'master'
        #elif iso.target == '2':
        #    target = 'slave'
        isotropy["subdevice_uid"] = f"pocam-led-master_{driver}-{emitter[-3:]}_{device}"
        isotropy["meas_time"] = iso.meas_time
        isotropy["meas_batch"] = batch






        grid_values = {}
        grid_values["data_format"] = "healpix"
        grid_values["projection"] = "mollview"

        if 'LMG' in iso.diode:
            grid_values["coarse"] = coarse
            grid_values["fine"] = fine
        elif 'KAPU' in iso.diode:
            grid_values["mode"] = mode                                         # str: either 'default' or 'fast'
        grid_values["power"] = pwm                                             # integer
        grid_values["temperature"] = temp                                      # number (pos. or neg.), e.g. 25, 0, -20, etc.
        grid_values["x_label"] =  "azimuth angle phi [°]"
        grid_values["y_label"] =  "zenith angle theta [°]"
        grid_values["z_label"] =  "relative emission intensity"
        grid_values["x_min"] = -np.pi
        grid_values["x_max"] = np.pi

        grid_values["y_min"] = -np.pi/2
        grid_values["y_max"] = np.pi/2

        grid_values["bins"] = len(iso.y_values)
        grid_values["z_values"] = iso.y_values        # array of normalized data

        grid_values["title"] = "angular-emission-profile"
        isotropy["meas_data"].append(grid_values)





        # "value" :
        value_iso = {}
        value_iso["data_format"] = "value"
        value_iso["value"] = round( iso.isotropy_value , 3 )
        value_iso["error"] = round( iso.isotropy_error , 3 )
        if 'LMG' in iso.diode:
            value_iso["coarse"] = coarse
            value_iso["fine"] = fine
        elif 'KAPU' in iso.diode:
            value_iso["mode"] = mode
        value_iso["power"] = pwm
        value_iso["temperature"] = temp
        value_iso["label"] = "Level-of-Isotropy-for-entire-Device"
        isotropy["meas_data"].append(value_iso)






        # We further add some comments and support files (links) to the json file:

        # the comments are describing the data but can be simply copy-pasted for all different L(E)Ds,
        # as they are not specifying on the exact data values:
        isotropy["comments"] = [
                                "The degree of isotropy is given as a value in [%]. It is obtained from combining the angular emission profiles of both hemispheres that are installed in the same POCAM and is calculated by normalizing the light emission data to the mean value, subtracting the such obtained minimum relative intensity from the maximum relative intensity and dividing it by 2",
                                f"This hemisphere {hemispheres[0]} is installed in POCAM {device} together with hemisphere {hemispheres[0]}",
                                "The phi-axis is the one that describes rotations around the long POCAM body axis, theta describes rotations perpendicular to phi-axis",
                                "The angular profile is measured in a constant solid angle approach, resulting in 192 measured angle tupples of azimuth and zenith",
                                "Azimuthal angle is scanned in the range of about 0-360° and zenith angle is scanned in the range of about 0-150°"
                                ]







    elif batch=='batch1':
        # Running the actual data inference:
        iso = isotropy_value_batch1(hemispheres=hemispheres,
                             diode= emitter)  


        ### take date of assembly of the device, as there will probably only one angular_emission_profile file per device
        ### more than one would mean, copy the same file in all subfolders

        ###### TALK TO MATT KAUER ABOUT THAT, also ask about subdevice_uid as there will be no master/slave


        # specifying some more key-value pairs for identification etc.:
        isotropy["device_uid"] = f"pocam-{iso.date1}_{device}"
        #if iso.target == '1':
        #    target = 'master'
        #elif iso.target == '2':
        #    target = 'slave'
        isotropy["subdevice_uid"] = f"pocam-led-master_{driver}-{emitter[-3:]}_{device}"
        isotropy["meas_time"] = iso.meas_time
        isotropy["meas_batch"] = batch



        grid_values = {}
        grid_values["data_format"] = "meshgrid"
        grid_values["projection"] = "mollview"

        if 'LMG' in iso.diode:
            grid_values["coarse"] = coarse
            grid_values["fine"] = fine
        elif 'KAPU' in iso.diode:
            grid_values["mode"] = mode                                         # str: either 'default' or 'fast'
        grid_values["power"] = pwm                                             # integer
        grid_values["temperature"] = temp                                      # number (pos. or neg.), e.g. 25, 0, -20, etc.
        grid_values["x_label"] =  "azimuth angle phi [°]"
        grid_values["y_label"] =  "zenith angle theta [°]"
        grid_values["z_label"] =  "relative emission intensity"
        grid_values["x_min"] = -np.pi
        grid_values["x_max"] = np.pi
        grid_values["x_bins"] = 6

        grid_values["y_min"] = -np.pi/2
        grid_values["y_max"] = np.pi/2
        grid_values["y_bins"] = 19

        grid_values["z_values"] = iso.y_values        # array of normalized data

        grid_values["title"] = "angular-emission-profile"
        isotropy["meas_data"].append(grid_values)





        # "value" :
        value_iso = {}
        value_iso["data_format"] = "value"
        value_iso["value"] = round( iso.isotropy_value , 3 )
        value_iso["error"] = round( iso.isotropy_error , 3 )
        if 'LMG' in iso.diode:
            value_iso["coarse"] = coarse
            value_iso["fine"] = fine
        elif 'KAPU' in iso.diode:
            value_iso["mode"] = mode
        value_iso["power"] = pwm
        value_iso["temperature"] = temp
        value_iso["label"] = "Level-of-Isotropy-for-entire-Device"
        isotropy["meas_data"].append(value_iso)



        # We further add some comments and support files (links) to the json file:

        # the comments are describing the data but can be simply copy-pasted for all different L(E)Ds,
        # as they are not specifying on the exact data values:
        isotropy["comments"] = [
                                "The degree of isotropy is given as a value in [%]. It is obtained from combining the angular emission profiles of both hemispheres that are installed in the same POCAM and is calculated by normalizing the light emission data to the mean value, subtracting the such obtained minimum relative intensity from the maximum relative intensity and dividing it by 2",
                                f"This hemisphere {hemispheres[0]} is installed in POCAM {device} together with hemisphere {hemispheres[0]}",
                                "The phi-axis is the one that describes rotations around the long POCAM body axis, theta describes rotations perpendicular to phi-axis",
                                "Azimuthal angle is scanned in increments of 60° covering the whole range of 0-360° and zenith angle is scanned in increments of 10° covering angles 0-150°"
                                ]





    else:
        print('[pocam is not in either of the two batches]')




    # end of loop




    # the support files contain one link where a POCAM_documentation file shall be uploaded,
    # explaining data taking, processing and interpretation in a more detailed way
    # but also contains the link to the specific raw data file(s) that is specified by the selected L(E)D
    isotropy["support_files"] = [
                                                {"filetype": "hdf5",
                                                 "hostname": "data.icecube.wisc.edu",
                                                 "pathname": f"/data/exp/IceCubeUpgrade/commissioning/pocam/pocam_{device}/{target}_hemisphere/{emitter}",
                                                 "comment" : "Here you can find the raw data, stored as an hdf5 file. For more detailed info on the structure and interpretation of the data files please consult the POCAM documentation guide."
                                                },
                                                {"filetype": "pdf",
                                                     "hostname": "data.icecube.wisc.edu",
                                                     "pathname": "/data/exp/IceCubeUpgrade/commissioning/pocam/POCAM_documentation.pdf",
                                                     "comment" : "Here you can find the POCAM documentation guide, for more detailed info on the data taking, data processing, interpretation, etc.."
                                                    }
                                        ]
    
    
    outfile1 = f'/home/ecp/database/data/{batch}/pocam_{device_id}/hem_{hemispheres[0]}/{emitter}/isotropy.json'

    with open(outfile1, 'w+') as file:
        json.dump(isotropy, file, indent=4)
        print(f"Dictionary saved to {outfile1}")
    
    
    outfile2 = f'/home/ecp/database/data/{batch}/pocam_{device_id}/hem_{hemispheres[1]}/{emitter}/isotropy.json'

    with open(outfile2, 'w+') as file:
        json.dump(isotropy, file, indent=4)
        print(f"Dictionary saved to {outfile2}")
    
    
    
    return isotropy
    


In [19]:
file_creation('018', 'LMG405')

Dictionary saved to /home/ecp/database/data/batch2/pocam_018/hem_35/LMG405/isotropy.json
Dictionary saved to /home/ecp/database/data/batch2/pocam_018/hem_36/LMG405/isotropy.json


{'device_uid': 'pocam-2025-04-16_09_08_41_018',
 'subdevice_uid': 'pocam-led-master_l-405_018',
 'meas_name': 'led-isotropy-level',
 'meas_class': 'display',
 'meas_stage': 'calibration',
 'meas_group': 'isotropy',
 'meas_site': 'tum',
 'meas_data': [{'data_format': 'healpix',
   'projection': 'mollview',
   'coarse': 1,
   'fine': 20,
   'power': 54000,
   'temperature': 25,
   'x_label': 'azimuth angle phi [°]',
   'y_label': 'zenith angle theta [°]',
   'z_label': 'relative emission intensity',
   'x_min': -3.141592653589793,
   'x_max': 3.141592653589793,
   'y_min': -1.5707963267948966,
   'y_max': 1.5707963267948966,
   'bins': 192,
   'z_values': [1.0039550136975044,
    0.9996401477166781,
    0.9971170652679411,
    1.003180259513372,
    1.0036474435145237,
    1.003362891037843,
    0.9939681039933571,
    0.9948113816516171,
    1.0015605663033393,
    0.9931985849601387,
    1.0017460915106706,
    1.0102482545438152,
    1.0022540757743825,
    1.003671110286578,
    0.99

In [12]:
#isotropy

In [ ]:
def plot_isotropy(hemispheres = ['03', '04'],
                  diode = 'LMG405',
                  path = None):
    
    """ This function is to plot the angular emission profile of two hemispheres.
        It either can be used to combine two hemispheres of one POCAM to display the complete emission profile of it or
        to virtually flip the emission profile of one hemisphere only. """

    # led = ['LMG405', 'LMG520', 'KAPU465', 'LMG450', 'KAPU405', 'LMG365']

    hem1 = f'/home/ecp/test_folder/prod_characterization_{hemispheres[0]}/cali_flange_{hemispheres[0]}'
    hem2 = f'/home/ecp/test_folder/prod_characterization_{hemispheres[1]}/cali_flange_{hemispheres[1]}'
    
    NSIDE = 2**2
    NPIX = hp.nside2npix(NSIDE)

    vec = hp.ang2vec(np.pi / 2, np.pi * 3 / 4)
    ipix_disc = hp.query_strip(NSIDE, np.radians(0), np.radians(150))
    deg = np.degrees(hp.pix2ang(nside=NSIDE, ipix=ipix_disc))

    
    x_pre = np.array([62.70,60.16,57.69,55.16,52.62,50.22,47.54,45.14,42.67,40.27,37.66,34.99,32.59,30.05,27.58,25.18,22.64,20.24,17.63,15.09,12.69,10.08,7.54,5.07,2.67,0.34])
    y_pre = np.array([0.960,0.966,0.968,0.969,0.971,0.975,0.980,0.981,0.990,0.990,0.991,0.991,0.994,0.997,1.000,1.000,1.000,1.003,1.001,1.004,1.004,1.006,1.006,1.006,1.006,1.007])
    x_values = np.array([65.17,67.64,70.18,70.11,72.64,72.58,75.18,75.25,77.64,77.65,80.18,80.18,82.64,82.64,85.10,85.10,87.64,87.63,90.10,90.16,90.26,92.62,95.09,95.08,97.56,97.55,100.16,100.01,102.55,102.54,105.01,105.00,107.55,107.54,110.01,110.07,112.61,112.54,115.08,115.15,117.62,117.62,119.95,119.95,125.,125.,130.,130.,135.,135.,140.,140.,150.,150.,160.,160.])
    y_values = np.array([0.958,0.953,0.939,0.949,0.909,0.934,0.874,0.903,0.832,0.860,0.783,0.798,0.725,0.726,0.668,0.656,0.606,0.577,0.542,0.497,0.480,0.416,0.416,0.337,0.353,0.262,0.293,0.186,0.233,0.122,0.176,0.069,0.125,0.025,0.075,0.005,0.039,-0.001,0.014,-0.001,0.006,-0.002,0.005,-0.002,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.])
    x_air = np.concatenate([np.flip(x_pre), x_values[::2]])[10:]
    x_ice = np.concatenate([np.flip(x_pre), x_values[1::2]])
    y_air = np.concatenate([np.flip(y_pre), y_values[::2]])[10:]
    y_ice = np.concatenate([np.flip(y_pre), y_values[1::2]])

    popt_air, _ = curve_fit(sigmoid, x_air, y_air, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)
    popt_ice, _ = curve_fit(sigmoid, x_ice, y_ice, p0 = [1.1, -1.0, 95, -0.001], maxfev = 1000)
    
    h1 = h5py.File(hem1, 'r')
    h2 = h5py.File(hem2, 'r')
    
    
    #led = ['LMG405', 'LMG365', 'LMG520', 'LMG450', 'KAPU405', 'KAPU465']
    led = [diode]
    
    
    for i,j in enumerate(led):
        string = 'angular_plots/'+j+'.png'
        fig = plt.figure()
        fig.set_size_inches(7,4)
        # print(j)

        data1 = np.array(h1.get(j)[2])
        data2 = np.array(h2.get(j)[2])


        # adjust azimuth angles
        mini1 = min(data1)
        mini2 = min(data2)

        y1 = np.array(data1)/mini1
        y2 = np.array(data2)/mini2

        y1 = y1*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)
        y2 = y2*sigmoid(deg[0],*popt_ice)/sigmoid(deg[0],*popt_air)

        y1 = np.concatenate((y1,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        y2 = np.concatenate((y2,np.array([0,0,0,0,0,0,0,0,0,0,0,0])), axis=None)
        
        
        y_mirror = np.flip(y2)
        y_total = (y1 + y_mirror)
        y_total = y_total/np.mean(y_total)
        
        zeniths = np.round(np.unique(deg[0]),1)
        azimuth = np.round(deg[1],2)


        zeniths = np.concatenate((np.array([0]), zeniths))


        indices = np.where(np.diff(azimuth) < 0)[0]
        azimuth_list = np.split(azimuth, indices+1)

        orig_map=plt.cm.get_cmap('Blues')
        rev = orig_map.reversed()
        isotropy_value = (np.max(y_total) - np.min(y_total))/2

        print(isotropy_value)
    
        hp.mollview(y_total, title = 'isotropic emission', cmap=rev, rot=-0)
        hp.graticule()
        plt.show()

        if path != None:
            #string = 'angular_plots/LMG405_real_adjusted_air.png'
            fig.savefig(path, dpi=700, bbox_inches='tight')   
        else:
            pass
        
        return(azimuth_list, zeniths)#, diff_data/mean)